# 🏃 App 1 — Tạo index khuôn mặt & số BIB (Colab/GPU)

Notebook này quét ảnh từ một hoặc nhiều folder Google Drive, sinh **embedding khuôn mặt**
và **đọc số BIB**, rồi xuất file `index.zip` cho từng folder.

**Cách dùng:**
1. **Runtime → Change runtime type → GPU**
2. Upload file `app1_indexer.zip` (chứa code ứng dụng) khi được hỏi
3. Chạy lần lượt 3 cell → giao diện hiện ra
4. Dán link/ID folder Drive → bấm **Bắt đầu quét**
5. Mỗi folder xong → tự tải `index.zip` về máy + lưu lên Drive (tuỳ chọn)

## 1. Upload code & cài thư viện (5–7 phút lần đầu)

In [ ]:
import os

REPO_URL = 'https://github.com/nguyenhoang1221hoangnguyen/bib-face-finder.git'
WORK     = '/content/bib-face-finder'
MARKER   = '/content/.deps_installed_v3'  # bump v4,v5,... nếu sau này đổi danh sách deps

# --- Clone hoặc pull repo ---
if not os.path.exists(WORK):
    !git clone -q "$REPO_URL" "$WORK"
else:
    !cd "$WORK" && git pull -q
os.chdir(WORK)

NEEDED = ['build_index.py', 'face_engine.py', 'bib_engine.py',
          'config.py', 'drive_source.py', 'bib_detector.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f'Thiếu file: {missing}'
print('✅ Code sẵn sàng tại:', os.getcwd())

# --- Cài deps (chỉ chạy 1 lần mỗi runtime) ---
if os.path.exists(MARKER):
    print('✅ Thư viện đã cài từ trước (skip).')
else:
    # Colab hiện ship CUDA 12 → onnxruntime-gpu phải >=1.19 (1.18 cần CUDA 11 libs).
    # PaddlePaddle GPU 2.6 cũng cần CUDA 11 → dùng CPU version (OCR không phải bottleneck).
    # faiss-cpu thay faiss-gpu (faiss-gpu trên PyPI không có wheel Python 3.11+).
    # Pin numpy<2 vì insightface/onnxruntime/paddle build với numpy 1.x ABI.
    !pip install -q \
      "numpy==1.26.4" \
      insightface==0.7.3 onnxruntime-gpu==1.19.2 faiss-cpu==1.8.0 \
      paddleocr==2.7.3 paddlepaddle==2.6.1 ultralytics==8.2.0 \
      opencv-python-headless==4.10.0.84 pandas==2.2.2 pyarrow==16.1.0 \
      google-api-python-client==2.134.0 google-auth==2.30.0 \
      python-dotenv==1.0.1 tqdm==4.66.4
    # Có dep khác có thể đã kéo numpy lên 2.x — ép lại 1.26.4.
    !pip install -q --force-reinstall --no-deps "numpy==1.26.4"
    open(MARKER, 'w').close()
    print('\n⚠️  Đã cài xong. KILL RUNTIME để load numpy đúng phiên bản...')
    print('    → Sau khi Colab tự reconnect, CHẠY LẠI CELL NÀY (sẽ skip cài).')
    os.kill(os.getpid(), 9)

!nvidia-smi -L
print('✅ Sẵn sàng.')

## 2. Cấu hình + đăng nhập Google

Khi chạy cell này, Colab sẽ mở popup xin quyền đọc Drive — bấm cho phép.

In [ ]:
import os, sys

# Sau khi kernel restart, CWD có thể bị reset về /content. Tự về lại repo dir.
WORK = '/content/bib-face-finder'
if os.getcwd() != WORK:
    os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

# ⬇️  DÁN ID FOLDER VÀO ĐÂY  ⬇️
DRIVE_FOLDER_ID = ''

ENABLE_BIB = True   # đặt False nếu chỉ cần khuôn mặt (nhanh hơn nhiều)

assert DRIVE_FOLDER_ID, 'Hãy điền DRIVE_FOLDER_ID phía trên rồi chạy lại cell này.'

from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default(scopes=['https://www.googleapis.com/auth/drive.readonly'])

os.environ['DRIVE_FOLDER_ID'] = DRIVE_FOLDER_ID
os.environ['FACE_CTX_ID']     = '0'                          # 0 = GPU
os.environ['ENABLE_BIB']      = 'true' if ENABLE_BIB else 'false'

from drive_source import DriveSource
from config import IndexerConfig
from build_index import build_index

drive  = DriveSource(credentials=creds)
config = IndexerConfig.from_env()


# Helper: chạy 1 hàm và đồng thời ghi toàn bộ stdout/stderr ra /content/run_log.txt
# (line-buffered → log vẫn còn ngay cả khi kernel segfault).
def run_logged(fn, *args, **kwargs):
    import traceback
    log_path = '/content/run_log.txt'
    log = open(log_path, 'w', buffering=1)
    orig_out, orig_err = sys.stdout, sys.stderr
    class _Tee:
        def __init__(self, *s): self.s = s
        def write(self, d):
            for x in self.s:
                x.write(d); x.flush()
        def flush(self):
            for x in self.s: x.flush()
    sys.stdout, sys.stderr = _Tee(orig_out, log), _Tee(orig_err, log)
    try:
        return fn(*args, **kwargs)
    except Exception:
        traceback.print_exc()
    finally:
        sys.stdout, sys.stderr = orig_out, orig_err
        log.close()
        print(f'\n📄 Log đã ghi vào {log_path} — chạy cell tải log ở cuối notebook nếu cần.')


print('✅ Sẵn sàng. Folder:', DRIVE_FOLDER_ID, '| BIB:', ENABLE_BIB)

## 3. Chạy thử 50 ảnh đầu (kiểm tra mọi thứ OK)

In [ ]:
run_logged(build_index, config, drive=drive, limit=50)

## 4. Chạy toàn bộ folder

Tốn từ vài phút đến vài giờ tuỳ số ảnh. Có thể đóng tab và mở lại — cell vẫn chạy ở phía Colab miễn là runtime chưa bị thu hồi.

In [ ]:
run_logged(build_index, config, drive=drive)

## 5. Tải index về máy

Sau khi tải `index.zip` xong, vào App 2 → trang quản lý → mở dự án → Upload zip này lên là xong.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('index', 'zip', 'data/index')
files.download('index.zip')

## 6. Tải log (chạy khi gặp lỗi)

Tải `run_log.txt` về thư mục `face-app/app1_indexer/` rồi nhắn "đọc log.txt" để Claude xem.

In [ ]:
from google.colab import files
files.download('/content/run_log.txt')